# ETL Project: Databricks notebook (Parts 4, 5, 6)
**Before running:** unzip `export_for_databricks.zip` from Colab, then upload the 5 `.parquet` files to the Volume created in cell 1 (Catalog, your schema, Volumes, `raw_files`, Upload to this volume). Change `SCHEMA` to your own name.

In [0]:

CATALOG = "workspace"
SCHEMA  = "humna_shop"      
VOL     = "raw_files"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOL}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
PATH = PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOL}/export_for_databricks"
display(dbutils.fs.ls(PATH))   # upload the 5 parquet files now, then rerun this cell to confirm

path,name,size,modificationTime
dbfs:/Volumes/workspace/humna_shop/raw_files/export_for_databricks/dim_category.parquet,dim_category.parquet,1863,1793759251000
dbfs:/Volumes/workspace/humna_shop/raw_files/export_for_databricks/dim_customer.parquet,dim_customer.parquet,38944,1793759251000
dbfs:/Volumes/workspace/humna_shop/raw_files/export_for_databricks/dim_date.parquet,dim_date.parquet,12908,1793759251000
dbfs:/Volumes/workspace/humna_shop/raw_files/export_for_databricks/fact_orders_incremental.parquet,fact_orders_incremental.parquet,88618,1793759251000
dbfs:/Volumes/workspace/humna_shop/raw_files/export_for_databricks/fact_orders_initial.parquet,fact_orders_initial.parquet,5746950,1793759257000


## Part 4: Delta tables, incremental load, time travel

In [0]:
spark.sql("DROP TABLE IF EXISTS fact_orders")

DataFrame[]

In [0]:
# ---- Load dimensions + FIRST load of fact_orders (this creates version 0). Run this cell only once. ----
from pyspark.sql import functions as F

for t in ["dim_date", "dim_customer", "dim_category"]:
    spark.read.parquet(f"{PATH}/{t}.parquet").write.format("delta").mode("overwrite").saveAsTable(t)

fo0 = (spark.read.parquet(f"{PATH}/fact_orders_initial.parquet")
       .withColumn("updated_at", F.col("updated_at").cast("timestamp")))
fo0.write.format("delta").mode("overwrite").saveAsTable("fact_orders")
print("fact_orders version 0 rows:", spark.table("fact_orders").count())

fact_orders version 0 rows: 197153


In [0]:
# ---- Incremental load: orders updated after 2026-10-20 are MERGED into fact_orders ----
from delta.tables import DeltaTable

inc = (spark.read.parquet(f"{PATH}/fact_orders_incremental.parquet")
       .withColumn("updated_at", F.col("updated_at").cast("timestamp")))
print("incremental rows:", inc.count())

(DeltaTable.forName(spark, "fact_orders").alias("t")
    .merge(inc.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

display(spark.sql("DESCRIBE HISTORY fact_orders"))     # <-- screenshot this for your README

incremental rows: 2847


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-04T03:31:34.000Z,78378717973429,hamnafine@gmail.com,MERGE,"Map(predicate -> [""(order_id#11576L = order_id#11537L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1809997843220236),7ef6976b-2fa7-406a-8fb3-ca03f110c68e,1104-032614-ejt5l4l4-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 53255, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 5174, materializeSourceTimeMs -> 769, numTargetRowsInserted -> 2847, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2560, numTargetRowsUpdated -> 0, numOutputRows -> 2847, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2847, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1743)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-04T03:31:17.000Z,78378717973429,hamnafine@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""313ad058-298f-4f3c-8202-1fae89936844"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-dd65df4c-6d97-416a-ab18-0e2f2ab8d498"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-4be70b5a-66c3-486e-bfa4-74dbab132f2a""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1809997843220236),f19569a4-2be2-4421-9177-930aacff39a0,1104-032614-ejt5l4l4-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 3596601, numDeletionVectorsRemoved -> 0, numOutputRows -> 197153, numOutputBytes -> 3543346)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-04T02:33:13.000Z,78378717973429,hamnafine@gmail.com,MERGE,"Map(predicate -> [""(order_id#13447L = order_id#13408L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1809997843220236),205cd07e-80d1-4757-b31f-6e78e21b6bd4,1104-021503-1d595243-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 53255, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3400, materializeSourceTimeMs -> 603, numTargetRowsInserted -> 2847, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1501, numTargetRowsUpdated -> 0, numOutputRows -> 2847, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2847, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1208)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13
1,2026-11-04T02:33:06.000Z,78378717973429,hamnafine@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(i

In [0]:
# ---- Time travel: version 0 vs current ----
v0  = spark.sql("SELECT COUNT(*) AS n FROM fact_orders VERSION AS OF 0").first()["n"]
now = spark.sql("SELECT COUNT(*) AS n FROM fact_orders").first()["n"]
print(f"version 0: {v0:,} rows | current: {now:,} rows | difference: {now - v0:,}")

version 0: 197,153 rows | current: 200,000 rows | difference: 2,847


## Part 5: gold tables
Rules used: revenue and average order value exclude `cancelled` and `returned` orders. City comes from `dim_customer` (the city the customer lived in when the order was placed). Write these rules in your README.

In [0]:
# ---- gold_monthly_sales (DataFrame API) ----
fo = spark.table("fact_orders"); dc = spark.table("dim_customer")
dd = spark.table("dim_date");    dcat = spark.table("dim_category")
valid = ~F.col("status").isin("cancelled", "returned")

gold_monthly_sales = (fo.filter(valid)
    .join(dc.select("customer_key", "city"), "customer_key")
    .join(dd.select("date_key", "year", "month"), "date_key")
    .withColumn("month_start", F.make_date("year", "month", F.lit(1)))
    .groupBy("month_start", "city")
    .agg(F.round(F.sum("amount"), 2).alias("revenue"),
         F.count("*").alias("order_count"),
         F.round(F.avg("amount"), 2).alias("avg_order_value")))
gold_monthly_sales.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_sales")
display(spark.table("gold_monthly_sales").orderBy("month_start", "city").limit(10))

month_start,city,revenue,order_count,avg_order_value
2025-01-01,Faisalabad,1.790322925E7,660,27126.1
2025-01-01,Hyderabad,7666129.55,381,20121.07
2025-01-01,Islamabad,2.156486435E7,955,22581.01
2025-01-01,Karachi,4.589267976E7,1999,22957.82
2025-01-01,Lahore,4.399671074E7,1763,24955.59
2025-01-01,Multan,1.256998366E7,546,23021.95
2025-01-01,Peshawar,1.223159699E7,533,22948.59
2025-01-01,Quetta,5053501.55,213,23725.36
2025-01-01,Rawalpindi,1.563838751E7,693,22566.22
2025-01-01,Sialkot,1.027927396E7,419,24532.87


In [0]:
# ---- gold_category_mix (Spark SQL) ----
spark.sql("""
CREATE OR REPLACE TABLE gold_category_mix AS
SELECT d.year, d.month, c.category,
       ROUND(SUM(f.amount), 2) AS revenue,
       COUNT(*)                AS order_count,
       ROUND(SUM(f.amount) / SUM(SUM(f.amount)) OVER (PARTITION BY d.year, d.month), 4) AS revenue_share
FROM fact_orders f
JOIN dim_date d     ON f.date_key = d.date_key
JOIN dim_category c ON f.category_key = c.category_key
WHERE f.status NOT IN ('cancelled', 'returned')
GROUP BY d.year, d.month, c.category
""")
display(spark.table("gold_category_mix").orderBy("year", "month", "category").limit(12))

year,month,category,revenue,order_count,revenue_share
2025,1,Beauty,9659881.13,1350,0.0501
2025,1,Books,4233955.13,1335,0.022
2025,1,Electronics,1.0706449692E8,1343,0.5553
2025,1,Fashion,1.894144038E7,1382,0.0982
2025,1,Grocery,6967827.8,1396,0.0361
2025,1,Home,4.592875596E7,1356,0.2382
2025,2,Beauty,8458816.0,1225,0.0498
2025,2,Books,3932550.42,1250,0.0232
2025,2,Electronics,9.172431014E7,1169,0.5405
2025,2,Fashion,1.44222626E7,1106,0.085


In [0]:
# ---- gold_customer_features (DataFrame API), as of 2026-10-24 (last day of data) ----
from pyspark.sql.window import Window
AS_OF = "2026-10-24"

fo = spark.table("fact_orders")
base = (fo.groupBy("customer_id")
    .agg(F.round(F.sum(F.when(valid, F.col("amount")).otherwise(0)), 2).alias("total_spend"),
         F.count("*").alias("n_orders"),
         F.max("order_date").alias("last_order_date"),
         F.round(F.avg(F.col("in_app").cast("int")), 3).alias("app_order_share"))
    .withColumn("days_since_last_order", F.datediff(F.lit(AS_OF).cast("date"), F.col("last_order_date"))))

w = Window.partitionBy("customer_id").orderBy(F.desc("n"), F.asc("category"))
fav = (fo.join(dcat, "category_key").groupBy("customer_id", "category").agg(F.count("*").alias("n"))
         .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
         .select("customer_id", F.col("category").alias("favourite_category")))

gold_customer_features = base.join(fav, "customer_id")
gold_customer_features.write.format("delta").mode("overwrite").saveAsTable("gold_customer_features")
print("customers:", spark.table("gold_customer_features").count())
display(spark.table("gold_customer_features").limit(10))

customers: 2000


customer_id,total_spend,n_orders,last_order_date,app_order_share,days_since_last_order,favourite_category
1,502690.99,46,2026-10-24,0.0,0,Grocery
2,1452276.42,56,2026-10-16,0.018,8,Electronics
3,1979487.34,67,2026-10-24,0.045,0,Electronics
4,514202.24,43,2026-09-20,0.07,34,Grocery
5,4106984.57,168,2026-10-24,0.018,0,Books
6,2981050.33,138,2026-10-22,0.043,2,Beauty
7,1056389.88,68,2026-10-17,0.059,7,Beauty
8,5262349.58,270,2026-10-22,0.033,2,Home
9,2412032.67,131,2026-10-17,0.053,7,Beauty
10,996239.27,39,2026-09-16,0.026,38,Electronics


### Part 5 answers (run each query, then write one sentence from the result)

In [0]:
# Q1. Which city grew fastest in 2026? (Jan-Sep 2026 revenue vs Jan-Sep 2025; October is partial)
display(spark.sql("""
SELECT city, r25, r26, ROUND((r26 / r25 - 1) * 100, 1) AS growth_pct
FROM (
  SELECT city,
         SUM(CASE WHEN YEAR(month_start) = 2025 THEN revenue END) AS r25,
         SUM(CASE WHEN YEAR(month_start) = 2026 THEN revenue END) AS r26
  FROM gold_monthly_sales
  WHERE MONTH(month_start) <= 9
  GROUP BY city)
WHERE r25 IS NOT NULL AND r26 IS NOT NULL
ORDER BY growth_pct DESC
"""))

city,r25,r26,growth_pct
Lahore,3.5364406037E8,3.7723226319E8,6.7
Karachi,4.2277640875999993E8,4.4560289507000005E8,5.4
Multan,1.1327085496000001E8,1.1785679122E8,4.0
Hyderabad,7.707422147E7,7.794858297999999E7,1.1
Faisalabad,1.4251176362E8,1.4312016685E8,0.4
Rawalpindi,1.4616391779000002E8,1.4432759358E8,-1.3
Peshawar,1.1343517868999998E8,1.0938244335000001E8,-3.6
Islamabad,1.9786265473E8,1.8398535845000005E8,-7.0
Sialkot,8.387209774000001E7,7.743136341999999E7,-7.7
Quetta,4.654126315E7,3.840729305E7,-17.5


In [0]:
# Q2. Do Lahore, Karachi and Islamabad get fewer orders on rainy days? (rainy = 1 mm or more)
display(spark.sql("""
SELECT c.city,
       CASE WHEN f.rain_mm >= 1 THEN 'rainy' ELSE 'dry' END AS day_type,
       COUNT(DISTINCT f.order_date) AS days,
       COUNT(*) AS orders,
       ROUND(COUNT(*) / COUNT(DISTINCT f.order_date), 1) AS orders_per_day
FROM fact_orders f
JOIN dim_customer c ON f.customer_key = c.customer_key
WHERE f.rain_mm IS NOT NULL
GROUP BY c.city, CASE WHEN f.rain_mm >= 1 THEN 'rainy' ELSE 'dry' END
ORDER BY c.city, day_type
"""))

city,day_type,days,orders,orders_per_day
Islamabad,dry,444,14852,33.5
Islamabad,rainy,218,7356,33.7
Karachi,dry,582,44397,76.3
Karachi,rainy,80,6213,77.7
Lahore,dry,502,32549,64.8
Lahore,rainy,160,10420,65.1


In [0]:
# Q3. Which category has the highest average order value?
display(spark.sql("""
SELECT category, ROUND(SUM(revenue) / SUM(order_count), 0) AS avg_order_value
FROM gold_category_mix
GROUP BY category
ORDER BY avg_order_value DESC
"""))

category,avg_order_value
Electronics,78494.0
Home,35105.0
Fashion,13437.0
Beauty,7067.0
Grocery,5335.0
Books,3227.0


In [0]:


1. Which city grew fastest in 2026?
Ans: Lahore grew fastest in 2026
2. Do Lahore, Karachi and Islamabad get fewer orders on rainy days?
Ans: No. Lahore, Karachi and Islamabad get about the same number of orders on rainy and dry days Lahore 65.1 vs 64.8, Karachi 77.7 vs 76.3, Islamabad 33.7 vs 33.5 orders per day, so rain does not reduce orders.
3. Which category has the highest average order value?
Ans: Electronics has the highest average order value that is  78494

## Part 6: churn model with MLflow
**Important:** with the required cutoff `2026-08-01`, almost no customer churns in this dataset (about 2 out of 2,000), so recall and precision will be meaningless. Run the notebook as required first. Then, as an extra experiment, set `CUTOFF = "2026-10-01"` (about 12% churn) and rerun this section to get a model that can actually learn. Mention this in your README.

In [0]:
spark.sql("USE workspace.humna_shop")

DataFrame[]

In [0]:
# ---- Features (only orders BEFORE the cutoff) + label ----
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CUTOFF = "2026-10-01"        # extra experiment (the assignment's cutoff is 2026-08-01)
END    = "2026-10-24"

fo  = spark.table("fact_orders")
cut = F.lit(CUTOFF).cast("date")

pre = fo.filter(F.col("order_date") < cut)
feat = (pre.groupBy("customer_id")
    .agg(F.count("*").alias("n_orders"),
         F.sum("amount").alias("total_spend"),
         F.avg("amount").alias("avg_order_value"),
         F.avg(F.col("in_app").cast("int")).alias("app_share"),
         F.max("order_date").alias("last_order"),
         F.sum(F.when(F.col("order_date") >= F.date_sub(cut, 90), 1).otherwise(0)).alias("orders_last_90d"))
    .withColumn("days_since_last_order", F.datediff(cut, F.col("last_order")))
    .drop("last_order"))

active = (fo.filter((F.col("order_date") >= cut) & (F.col("order_date") <= F.lit(END).cast("date")))
            .select("customer_id").distinct().withColumn("active", F.lit(1)))

churn_features = (feat.join(active, "customer_id", "left")
    .withColumn("churned", F.when(F.col("active").isNull(), 1).otherwise(0)).drop("active"))
churn_features.write.format("delta").mode("overwrite").saveAsTable("churn_features")

FEATURES_VERSION = DeltaTable.forName(spark, "churn_features").history(1).select("version").first()[0]
pdf = spark.table("churn_features").toPandas()
print("rows:", len(pdf), "| churned:", int(pdf["churned"].sum()), f"({pdf['churned'].mean():.1%}) | features Delta version:", FEATURES_VERSION)

rows: 2000 | churned: 242 (12.1%) | features Delta version: 1


In [0]:
# ---- Train 3 settings, log each run in MLflow ----
import mlflow, mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

FEATURES = ["n_orders", "total_spend", "avg_order_value", "app_share", "orders_last_90d", "days_since_last_order"]
X, y = pdf[FEATURES], pdf["churned"]
strat = y if y.sum() >= 2 else None
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=strat)

experiments = [
    ("logreg",    LogisticRegression(max_iter=2000, class_weight="balanced"),
                  {"model": "LogisticRegression", "class_weight": "balanced"}),
    ("rf_depth3", RandomForestClassifier(n_estimators=100, max_depth=3, class_weight="balanced", random_state=42),
                  {"model": "RandomForest", "n_estimators": 100, "max_depth": 3, "class_weight": "balanced"}),
    ("rf_depth8", RandomForestClassifier(n_estimators=100, max_depth=8, class_weight="balanced", random_state=42),
                  {"model": "RandomForest", "n_estimators": 100, "max_depth": 8, "class_weight": "balanced"}),
]

for run_name, model, params in experiments:
    with mlflow.start_run(run_name=run_name) as run:
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        mlflow.log_params(params)
        mlflow.log_param("cutoff", CUTOFF)
        mlflow.log_param("features_delta_version", FEATURES_VERSION)
        mlflow.log_metric("accuracy",  accuracy_score(yte, pred))
        mlflow.log_metric("recall",    recall_score(yte, pred, zero_division=0))
        mlflow.log_metric("precision", precision_score(yte, pred, zero_division=0))
        mlflow.log_metric("f1",        f1_score(yte, pred, zero_division=0))
        mlflow.sklearn.log_model(model, artifact_path="model")
        print(run_name, "| run_id:", run.info.run_id)

2026/11/04 19:14:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-3da1f8b9-c627.cloud.databricks.com/ml/experiments/1809997843220236/models/m-9315fed0040f4001aae2d8c92104bc0d?o=7474645955067091
2026/11/04 19:14:46 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


logreg | run_id: 013abe5c8bea4de7bfdd0c3c4b1674fa


2026/11/04 19:14:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-3da1f8b9-c627.cloud.databricks.com/ml/experiments/1809997843220236/models/m-7f6c05c645954361b447ecf8aea7dce1?o=7474645955067091
2026/11/04 19:14:51 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


rf_depth3 | run_id: e4cc812652994b1c990819f7c53452d4


2026/11/04 19:14:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-3da1f8b9-c627.cloud.databricks.com/ml/experiments/1809997843220236/models/m-5681bfe4940f4876b35b9f3809b5d18a?o=7474645955067091
2026/11/04 19:14:56 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


rf_depth8 | run_id: ecac2feb108a4456928d44f2ebf5b518


In [0]:
# ---- Compare runs (also open the Experiments UI and screenshot the comparison) ----
runs = mlflow.search_runs(order_by=["metrics.recall DESC"])
cols = ["run_id", "tags.mlflow.runName", "metrics.accuracy", "metrics.recall", "metrics.precision", "metrics.f1", "params.features_delta_version"]
display(runs[[c for c in cols if c in runs.columns]].head(6))

run_id,tags.mlflow.runName,metrics.accuracy,metrics.recall,metrics.precision,metrics.f1,params.features_delta_version
013abe5c8bea4de7bfdd0c3c4b1674fa,logreg,0.556,0.7166666666666667,0.17338709677419356,0.2792207792207792,1
e4cc812652994b1c990819f7c53452d4,rf_depth3,0.558,0.7,0.17142857142857143,0.2754098360655738,1
ecac2feb108a4456928d44f2ebf5b518,rf_depth8,0.694,0.38333333333333336,0.16546762589928057,0.23115577889447236,1
0cced493498a4e6caf8e6a03e7e8d464,rf_depth8,1.0,0.0,0.0,0.0,0
f993b8cb64f348c8a4808c0ff47f3eea,rf_depth3,1.0,0.0,0.0,0.0,0
d8fbbec64c2e4332b599e3e78a211371,logreg,0.954,0.0,0.0,0.0,0


**Your choice (write 2 to 3 sentences in the README).** For a win-back campaign, **recall** matters most: it is the share of customers who really churn that the model catches, and missing a leaving customer costs more than sending an offer to someone who would have stayed. Check precision too, so marketing is not flooded with wrong targets. Pick the run with the best recall that still has acceptable precision.

In [0]:
# ---- Reload the chosen model by run ID and score 5 customers ----
best_run_id = runs.sort_values("metrics.recall", ascending=False).iloc[0]["run_id"]   # or paste the run_id you chose
model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

sample = pdf.sample(5, random_state=1)
out = sample[["customer_id"]].copy()
out["churn_probability"] = model.predict_proba(sample[FEATURES])[:, 1].round(3)
out["predicted_churn"]   = model.predict(sample[FEATURES])
out["actual_churned"]    = sample["churned"].values
display(out)

customer_id,churn_probability,predicted_churn,actual_churned
1105,0.411,0,0
1761,0.516,1,0
715,0.64,1,1
1248,0.003,0,0
570,0.329,0,0


In [0]:
spark.sql("USE workspace.humna_shop")
display(spark.sql("DESCRIBE HISTORY fact_orders"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-04T03:31:34.000Z,78378717973429,hamnafine@gmail.com,MERGE,"Map(predicate -> [""(order_id#11576L = order_id#11537L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1809997843220236),7ef6976b-2fa7-406a-8fb3-ca03f110c68e,1104-032614-ejt5l4l4-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 53255, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 5174, materializeSourceTimeMs -> 769, numTargetRowsInserted -> 2847, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2560, numTargetRowsUpdated -> 0, numOutputRows -> 2847, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2847, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1743)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-04T03:31:17.000Z,78378717973429,hamnafine@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""313ad058-298f-4f3c-8202-1fae89936844"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-dd65df4c-6d97-416a-ab18-0e2f2ab8d498"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-4be70b5a-66c3-486e-bfa4-74dbab132f2a""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1809997843220236),f19569a4-2be2-4421-9177-930aacff39a0,1104-032614-ejt5l4l4-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 3596601, numDeletionVectorsRemoved -> 0, numOutputRows -> 197153, numOutputBytes -> 3543346)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-04T02:33:13.000Z,78378717973429,hamnafine@gmail.com,MERGE,"Map(predicate -> [""(order_id#13447L = order_id#13408L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1809997843220236),205cd07e-80d1-4757-b31f-6e78e21b6bd4,1104-021503-1d595243-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 53255, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3400, materializeSourceTimeMs -> 603, numTargetRowsInserted -> 2847, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1501, numTargetRowsUpdated -> 0, numOutputRows -> 2847, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2847, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1208)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13
1,2026-11-04T02:33:06.000Z,78378717973429,hamnafine@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(i

In [0]:
spark.sql("USE workspace.humna_shop")
display(spark.sql("""
SELECT version, timestamp, operation
FROM (DESCRIBE HISTORY fact_orders)
ORDER BY version DESC
"""))

version,timestamp,operation
4,2026-11-04T03:31:34.000Z,MERGE
3,2026-11-04T03:31:17.000Z,CREATE OR REPLACE TABLE AS SELECT
2,2026-11-04T02:33:13.000Z,MERGE
1,2026-11-04T02:33:06.000Z,CREATE OR REPLACE TABLE AS SELECT
0,2026-11-04T02:31:09.000Z,CREATE OR REPLACE TABLE AS SELECT
